## Import packages

In [ ]:
import sys
sys.path.append('..')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from src import preprocess_single_image
from src import evaluate_model, plot_confusion_matrix, plot_roc_curve
from src import create_data_generators
from src import make_gradcam_heatmap, display_gradcam

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Load best model

In [ ]:
model = load_model('../models/best_model.keras')

with open('../models/best_config.json', 'r') as f:
    config = json.load(f)

print(f"Loaded best model: {config['source']}")
print(f"Accuracy: {config['accuracy'] * 100:.2f}%")

# Determine image size based on model
# VGG16 uses 150x150, ResNet50/Baseline use 224x224
if '150' in config['source']:
    target_size = (150, 150)
else:
    target_size = (224, 224)

print(f"Image size: {target_size[0]}x{target_size[1]}")
model.summary()

## Define class names and prediction helper

In [ ]:
class_names = ['NORMAL', 'PNEUMONIA']

def predict_xray(img_path, model, target_size=(150, 150)):
    """Predict and display result for a single X-ray image."""
    img_batch, img_original = preprocess_single_image(img_path, target_size=target_size)
    prediction = model.predict(img_batch, verbose=0)[0][0]

    pred_class = 1 if prediction >= 0.5 else 0
    pred_label = class_names[pred_class]
    confidence = prediction if pred_class == 1 else 1 - prediction

    plt.figure(figsize=(6, 6))
    plt.imshow(img_original / 255.0, cmap='gray')
    color = 'red' if pred_label == 'PNEUMONIA' else 'green'
    plt.title(f'Prediction: {pred_label} ({confidence:.1%})', fontsize=14, color=color)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return prediction


## Predict on sample Normal images

In [ ]:
data_dir = '../data/chest_xray'
test_normal_dir = os.path.join(data_dir, 'test', 'NORMAL')
normal_images = [f for f in os.listdir(test_normal_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][:4]

print("PREDICTIONS ON NORMAL X-RAYS:")
print("-" * 40)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_name in zip(axes, normal_images):
    img_path = os.path.join(test_normal_dir, img_name)
    img_batch, img_original = preprocess_single_image(img_path, target_size=target_size)
    pred = model.predict(img_batch, verbose=0)[0][0]

    pred_label = 'PNEUMONIA' if pred >= 0.5 else 'NORMAL'
    confidence = pred if pred >= 0.5 else 1 - pred
    color = 'red' if pred_label == 'PNEUMONIA' else 'green'

    ax.imshow(img_original / 255.0, cmap='gray')
    ax.set_title(f'{pred_label}\n({confidence:.1%})', fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('Normal X-Ray Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## Predict on sample Pneumonia images

In [ ]:
test_pneumonia_dir = os.path.join(data_dir, 'test', 'PNEUMONIA')
pneumonia_images = [f for f in os.listdir(test_pneumonia_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][:4]

print("PREDICTIONS ON PNEUMONIA X-RAYS:")
print("-" * 40)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_name in zip(axes, pneumonia_images):
    img_path = os.path.join(test_pneumonia_dir, img_name)
    img_batch, img_original = preprocess_single_image(img_path, target_size=target_size)
    pred = model.predict(img_batch, verbose=0)[0][0]

    pred_label = 'PNEUMONIA' if pred >= 0.5 else 'NORMAL'
    confidence = pred if pred >= 0.5 else 1 - pred
    color = 'red' if pred_label == 'PNEUMONIA' else 'green'

    ax.imshow(img_original / 255.0, cmap='gray')
    ax.set_title(f'{pred_label}\n({confidence:.1%})', fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('Pneumonia X-Ray Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## Grad-CAM visualization on Normal images

In [ ]:
print("GRAD-CAM: Where the model looks (NORMAL)")
print("-" * 40)

for img_name in normal_images[:2]:
    img_path = os.path.join(test_normal_dir, img_name)
    img_batch, img_original = preprocess_single_image(img_path, target_size=target_size)

    prediction = model.predict(img_batch, verbose=0)[0][0]
    heatmap = make_gradcam_heatmap(img_batch, model)
    display_gradcam(img_original, heatmap, prediction, class_names)



## Grad-CAM visualization on Pneumonia images

In [ ]:
print("GRAD-CAM: Where the model looks (PNEUMONIA)")
print("-" * 40)

for img_name in pneumonia_images[:2]:
    img_path = os.path.join(test_pneumonia_dir, img_name)
    img_batch, img_original = preprocess_single_image(img_path, target_size=target_size)

    prediction = model.predict(img_batch, verbose=0)[0][0]
    heatmap = make_gradcam_heatmap(img_batch, model)
    display_gradcam(img_original, heatmap, prediction, class_names)


## Batch prediction on full test set

In [ ]:
_, _, test_gen = create_data_generators(
    data_dir, target_size=target_size, batch_size=32, augment=False
)

results = evaluate_model(model, test_gen)
plot_confusion_matrix(
    results['y_true'], results['y_pred'],
    results['class_names'],
    title="Best Model - Test Set Confusion Matrix"
)


## Final dashboard

In [ ]:
# Load all results
with open('../models/baseline_results.json', 'r') as f:
    baseline = json.load(f)
with open('../models/transfer_results.json', 'r') as f:
    resnet = json.load(f)

# Try loading VGG16 results
try:
    with open('../models/vgg16_results.json', 'r') as f:
        vgg16 = json.load(f)
    has_vgg16 = True
except FileNotFoundError:
    has_vgg16 = False

print("\n")
print("╔" + "═" * 58 + "╗")
print("║" + "  PNEUMONIA DETECTION DASHBOARD".ljust(58) + "║")
print("╠" + "═" * 58 + "╣")
print("║" + f"  Best Model: {config['source']}".ljust(58) + "║")
print("║" + f"  Image Size: {target_size[0]}x{target_size[1]}".ljust(58) + "║")
print("║" + f"  Test Accuracy: {results['accuracy'] * 100:.2f}%".ljust(58) + "║")
print("║" + f"  Precision: {results['precision']:.4f}".ljust(58) + "║")
print("║" + f"  Recall: {results['recall']:.4f}".ljust(58) + "║")
print("║" + f"  F1-Score: {results['f1']:.4f}".ljust(58) + "║")
print("╠" + "═" * 58 + "╣")
print("║" + "  MODEL COMPARISON".ljust(58) + "║")
print("║" + f"  Baseline CNN:        {baseline['accuracy'] * 100:.2f}%".ljust(58) + "║")
print("║" + f"  ResNet50 Transfer:   {resnet['accuracy'] * 100:.2f}%".ljust(58) + "║")
if has_vgg16:
    print("║" + f"  VGG16 Transfer:      {vgg16['accuracy'] * 100:.2f}%".ljust(58) + "║")
print("╠" + "═" * 58 + "╣")
print("║" + f"  Test samples: {len(results['y_true'])}".ljust(58) + "║")
print("║" + f"  Correct: {np.sum(results['y_true'] == results['y_pred'])}".ljust(58) + "║")
misclassified = np.sum(results['y_true'] != results['y_pred'])
print("║" + f"  Misclassified: {misclassified}".ljust(58) + "║")
print("║" + f"  Improvement over baseline: {(results['accuracy'] - baseline['accuracy']) * 100:+.2f}%".ljust(58) + "║")
print("╚" + "═" * 58 + "╝")